In [1]:
import random
import sys

_previous_colors = []

def cprint(*args):
    global _previous_colors

    while True:
        bg = tuple(random.randint(100, 220) for _ in range(3))
        r, g, b = bg

        luminance = 0.2126*r + 0.7152*g + 0.0722*b
        fg = (20, 20, 20) if luminance > 140 else (255, 255, 255)

        color = (fg, bg)

        if color not in _previous_colors:
            break

    _previous_colors.append(color)
    _previous_colors[:] = _previous_colors[-2:]

    fr, f_g, fb = fg
    br, bg_g, bb = bg

    text = "".join(map(str, args))

    sys.stdout.write(
        f"\033[38;2;{fr};{f_g};{fb}m"
        f"\033[48;2;{br};{bg_g};{bb}m"
        f"{text}\033[0m"
    )
    sys.stdout.flush()

In [3]:
import ollama

WIDTH = 80

while True:
    prompt = input("\n> ")

    if not prompt.strip() or prompt.lower() in ("exit", "quit"):
        break

    print("\nYou:")
    print(prompt)
    print("\nQwen:")

    stream = ollama.chat(
        model="qwen2.5:latest",
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    word = ""
    column = 0

    def flush_word():
        global column
        if not word:
            return

        if column and column + len(word) > WIDTH:
            cprint("\n")
            column = 0

        cprint(word)
        column += len(word)

    for chunk in stream:
        for char in chunk["message"]["content"]:
            if char.isspace():
                flush_word()
                word = ""

                if char == "\n":
                    cprint("\n")
                    column = 0
                elif column < WIDTH:
                    cprint(" ")
                    column += 1
            else:
                word += char

    flush_word()
    cprint("\n")


You:
Hello there

Qwen:
Hello! How can I assist you today?

You:
Write basic code for a transformer (pytorch)

Qwen:
Certainly! Below is a basic implementation of a transformer model using PyTorch.
This example will create a simple transformer model with a single encoder and 
decoder layer. For simplicity, we'll use the `torch.nn` module to define the 
layers.

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
(-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe